In [2]:
import pyspark
from pyspark.sql import SparkSession

In [10]:
!java --version

openjdk 25.0.1 2025-10-21 LTS
OpenJDK Runtime Environment Microsoft-12574222 (build 25.0.1+8-LTS)
OpenJDK 64-Bit Server VM Microsoft-12574222 (build 25.0.1+8-LTS, mixed mode, sharing)


In [11]:
!export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

In [12]:
!export PATH="${JAVA_HOME}/bin:${PATH}"

In [13]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('read‑parquet') \
    .getOrCreate()

In [16]:

from pyspark import SparkFiles

file_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet'
spark.sparkContext.addFile(file_url)
df = spark.read.csv(SparkFiles.get('yellow_tripdata_2025-11.parquet'), header=True)

df.count()

26/03/05 19:58:57 WARN SparkContext: The path https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet has been added already. Overwriting of added paths is not supported in the current version.
26/03/05 19:58:57 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /tmp/spark-080ed586-84ee-4c71-ac02-96c27f3c58d9/userFiles-aafb6eec-db81-4788-8878-4b6ef332d452/yellow_tripdata_2025-11.parquet.
java.lang.UnsupportedOperationException: getSubject is not supported
	at java.base/javax.security.auth.Subject.getSubject(Subject.java:277)
	at org.apache.hadoop.security.UserGroupInformation.getCurrentUser(UserGroupInformation.java:588)
	at org.apache.hadoop.fs.FileSystem$Cache$Key.<init>(FileSystem.java:3888)
	at org.apache.hadoop.fs.FileSystem$Cache$Key.<init>(FileSystem.java:3878)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3666)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apac

UnsupportedOperationException: getSubject is not supported

In [ ]:
import pandas as pd

df_pandas = pd.read_parquet('yellow_tripdata_2025-11.parquet')
df_pandas.dtypes


VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object

In [3]:
df_zone_pandas = pd.read_csv('taxi_zone_lookup.csv')
df_zone_pandas.dtypes

LocationID       int64
Borough         object
Zone            object
service_zone    object
dtype: object

In [6]:
from pyspark.sql import types
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("passenger_count", types.DoubleType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.DoubleType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [9]:
df_spark = spark.createDataFrame(df_pandas)

26/03/04 14:11:42 WARN FileSystem: Cannot load filesystem
java.util.ServiceConfigurationError: org.apache.hadoop.fs.FileSystem: Provider org.apache.hadoop.fs.viewfs.ViewFileSystem could not be instantiated
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:552)
	at java.base/java.util.ServiceLoader$ProviderImpl.newInstance(ServiceLoader.java:712)
	at java.base/java.util.ServiceLoader$ProviderImpl.get(ServiceLoader.java:672)
	at java.base/java.util.ServiceLoader$2.next(ServiceLoader.java:1256)
	at org.apache.hadoop.fs.FileSystem.loadFileSystems(FileSystem.java:3525)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3562)
	at org.apache.hadoop.fs.FsUrlStreamHandlerFactory.<init>(FsUrlStreamHandlerFactory.java:77)
	at org.apache.spark.sql.internal.SharedState$.liftedTree2$1(SharedState.scala:209)
	at org.apache.spark.sql.internal.SharedState$.org$apache$spark$sql$internal$SharedState$$setFsUrlStreamHandlerFactory(SharedState.scala:208)
	at org.apache.spark.

In [1]:
df_spark = df_spark.repartition(24)
# df_spark.write.parquet('yellow_tripdata_2025-11_spark.parquet', mode='overwrite')

NameError: name 'df_spark' is not defined

In [6]:
trips_nov15 = df_pandas[
    df_pandas["tpep_pickup_datetime"].dt.date == pd.to_datetime("2025-11-15").date()
]

count = len(trips_nov15)
print("Number of trips on Nov 15:", count)

Number of trips on Nov 15: 162604


In [13]:
from pyspark.sql import functions as F

trips_nov15 = df_spark.filter(
    F.to_date("tpep_pickup_datetime") == F.lit("2025-11-15")
)

count = trips_nov15.count()

print("Number of trips on Nov 15:", count)

26/03/04 14:32:10 ERROR Inbox: An error happened while processing message in the inbox for LocalSchedulerBackendEndpoint
java.lang.OutOfMemoryError: Java heap space
	at java.base/java.util.Arrays.copyOf(Arrays.java:3537)
	at java.base/java.io.ByteArrayOutputStream.ensureCapacity(ByteArrayOutputStream.java:100)
	at java.base/java.io.ByteArrayOutputStream.write(ByteArrayOutputStream.java:132)
	at org.apache.spark.util.ByteBufferOutputStream.write(ByteBufferOutputStream.scala:41)
	at java.base/java.io.ObjectOutputStream$BlockDataOutputStream.drain(ObjectOutputStream.java:1765)
	at java.base/java.io.ObjectOutputStream$BlockDataOutputStream.setBlockDataMode(ObjectOutputStream.java:1674)
	at java.base/java.io.ObjectOutputStream.writeObject0(ObjectOutputStream.java:1090)
	at java.base/java.io.ObjectOutputStream.writeObject(ObjectOutputStream.java:325)
	at org.apache.spark.serializer.JavaSerializationStream.writeObject(JavaSerializer.scala:47)
	at org.apache.spark.serializer.JavaSerializerInst

KeyboardInterrupt: 

In [7]:
longest_trip_hours = (
    (df_pandas["tpep_dropoff_datetime"] - df_pandas["tpep_pickup_datetime"])
    .dt.total_seconds()
    .div(3600)
    .max()
)

print(longest_trip_hours)

90.64666666666666


In [4]:
df_joined = df_pandas.merge(
    df_zone_pandas,
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
)
zone_counts = (
    df_joined
    .groupby("Zone")
    .size()
    .reset_index(name="trip_count")
    .sort_values("trip_count")
)

zone_counts.head()

,Zone,trip_count
79,Eltingville/Annadale/Prince's Bay,1
97,Governor's Island/Ellis Island/Liberty Island,1
2,Arden Heights,1
181,Port Richmond,3
102,Green-Wood Cemetery,4


In [5]:
least_zone = zone_counts.iloc[0]
print(least_zone)

Zone          Eltingville/Annadale/Prince's Bay
trip_count                                    1
Name: 79, dtype: object
